In [6]:
! pip install streamlit -q
! pip install joblib -q
!pip install streamlit joblib pyngrok -q

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib

st.set_page_config(page_title="IDx Partners — Credit Risk & IFRS9 Simulator", layout="wide")

# ==============================
# LOAD PIPELINE (hasil notebook final: preprocessor + calibrated_model + lgd_by_grade)
# ==============================
MODEL_PATH = "/content/ifrs9_final_output (1).pkl"   

@st.cache_resource
def load_pipeline():
    return joblib.load(MODEL_PATH)

pipeline = load_pipeline()
preprocessor      = pipeline['preprocessor']
calibrated_model  = pipeline['calibrated_model']
lgd_by_grade      = pipeline['lgd_by_grade']     # dict, misal {'A':0.5746, 'B':0.6100, ...}
lgd_overall       = pipeline['lgd_overall']
feature_cols      = pipeline['feature_cols']

# ==============================
# TITLE
# ==============================
st.title("💳 Credit Risk & Profitability Simulator — IDx Partners")
st.markdown("""
Simulasi keputusan lending untuk **1 pengajuan baru**, berdasarkan **PD (model XGBoost terkalibrasi)** ×
**LGD (asumsi bisnis, bisa digeser)** × **EAD**. Model & LGD di sini hasil pipeline notebook final
(leakage-free split, kalibrasi isotonic).
""")

# ==============================
# SIDEBAR INPUT
# ==============================
st.sidebar.header("🧾 Borrower Input")

loan_amount   = st.sidebar.number_input("Loan Amount ($)", min_value=500, max_value=35000, value=12000, step=500)
interest_rate = st.sidebar.slider("Interest Rate (%)", 5.0, 30.0, 13.65)
term          = st.sidebar.selectbox("Loan Term (months)", [36, 60])
grade         = st.sidebar.selectbox("Grade", ['A', 'B', 'C', 'D', 'E', 'F', 'G'], index=1)
annual_inc    = st.sidebar.number_input("Annual Income ($)", min_value=0, value=61920, step=1000)
dti           = st.sidebar.number_input("Debt-to-Income Ratio (%)", min_value=0.0, value=16.05)

emp_length_ui = st.sidebar.selectbox("Employment Length", [
    "< 1 year", "1 year", "2 years", "3 years", "4 years", "5 years",
    "6 years", "7 years", "8 years", "9 years", "10+ years", "Unknown"
])
home_ownership = st.sidebar.selectbox("Home Ownership", ['RENT', 'OWN', 'MORTGAGE', 'OTHER', 'NONE', 'ANY'])

emp_map = {"< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3, "4 years": 4, "5 years": 5,
           "6 years": 6, "7 years": 7, "8 years": 8, "9 years": 9, "10+ years": 10, "Unknown": -1}
emp_length = emp_map[emp_length_ui]
emp_length_missing = 1 if emp_length_ui == "Unknown" else 0

# Default LGD diambil dari lookup historis per grade (hasil notebook final) --
# tetap bisa digeser manual, ini justru inti simulator: LGD asumsi bisnis vs PD model.
default_lgd = float(lgd_by_grade.get(grade, lgd_overall))
lgd = st.sidebar.slider(
    "LGD (Loss Given Default)", 0.0, 1.0, default_lgd,
    help=f"Default dari LGD historis grade {grade} ({default_lgd:.1%}, dihitung dari populasi train notebook final). Geser buat sensitivity analysis."
)

# ==============================
# BANGUN INPUT SESUAI FEATURE_COLS MODEL
# ==============================
# Nilai default (median/mode dari populasi training notebook final) untuk fitur
# yang tidak dikontrol lewat UI -- biar prediksi tetap representatif & konsisten.
DEFAULTS = {
    'sub_grade': 'B3', 'verification_status': 'Verified', 'purpose': 'debt_consolidation',
    'addr_state': 'CA', 'initial_list_status': 'f',
    'delinq_2yrs': 0.0, 'inq_last_6mths': 1.0, 'open_acc': 10.0, 'pub_rec': 0.0,
    'revol_bal': 10979.0, 'revol_util': 56.5, 'total_acc': 23.0,
    'collections_12_mths_ex_med': 0.0, 'acc_now_delinq': 0.0,
    'tot_coll_amt': 0.0, 'tot_cur_bal': 80282.0, 'total_rev_hi_lim': 22100.0,
}

# installment dihitung ulang (amortisasi) biar konsisten sama loan_amount/term/int_rate
monthly_rate = (interest_rate / 100) / 12
installment = loan_amount * monthly_rate * (1 + monthly_rate) ** term / ((1 + monthly_rate) ** term - 1)

input_row = {
    'loan_amnt': loan_amount, 'term': term, 'int_rate': interest_rate, 'installment': installment,
    'grade': grade, 'emp_length': emp_length, 'emp_length_missing': emp_length_missing,
    'home_ownership': home_ownership, 'annual_inc': annual_inc, 'dti': dti,
    **DEFAULTS
}
input_df = pd.DataFrame([input_row])[feature_cols + ['emp_length_missing']]

# ==============================
# PREDICT PD (model terkalibrasi — bukan model mentah)
# ==============================
try:
    input_processed = preprocessor.transform(input_df)
    pd_lifetime = calibrated_model.predict_proba(input_processed)[0, 1]
except Exception as e:
    st.error(f"Model error: {e}")
    st.stop()

# PD 12-bulan: approksimasi dari PD lifetime pakai constant hazard rate (sama seperti notebook final)
monthly_hazard = 1 - (1 - pd_lifetime) ** (1 / term)
pd_12m = 1 - (1 - monthly_hazard) ** 12

# ==============================
# ECONOMICS — ECL eksplisit: ECL = PD x LGD x EAD
# ==============================
# EAD di simulator ini = loan_amount (facility penuh), BUKAN out_prncp seperti di notebook
# portofolio historis -- karena ini pengajuan BARU yang belum cair, jadi belum ada
# outstanding balance yang bisa dikurangi. Beda konteks, beda definisi EAD yang tepat.
EAD = loan_amount
interest_income = loan_amount * (interest_rate / 100)

ecl_lifetime = pd_lifetime * lgd * EAD
ecl_12m = pd_12m * lgd * EAD
expected_profit = (1 - pd_lifetime) * interest_income - ecl_lifetime

decision = "APPROVE ✅" if expected_profit > 0 else "REJECT ❌"

# ==============================
# DISPLAY
# ==============================
st.markdown("## 📊 Decision Summary")

col1, col2, col3, col4 = st.columns(4)
col1.metric("PD (lifetime, terkalibrasi)", f"{pd_lifetime:.2%}")
col2.metric("LGD", f"{lgd:.2%}")
col3.metric("ECL Lifetime", f"${ecl_lifetime:,.0f}")
col4.metric("Expected Profit", f"${expected_profit:,.0f}")

st.caption(f"ECL = PD × LGD × EAD  |  ECL 12-bulan (approksimasi): ${ecl_12m:,.0f}  |  PD 12-bulan: {pd_12m:.2%}")

st.markdown("---")
if expected_profit > 0:
    st.success(f"### {decision}")
else:
    st.error(f"### {decision}")

# ==============================
# INSIGHT
# ==============================
st.markdown("## 🧠 Insight")
if expected_profit < 0:
    if pd_lifetime < 0.3:
        st.warning("Risiko default relatif rendah, tapi LGD tinggi bikin pengajuan ini tidak profitable.")
    else:
        st.warning("Risiko default tinggi + LGD tinggi = kerugian.")
else:
    st.success("Profitable di bawah asumsi saat ini.")

# ==============================
# PANEL: LIMITASI MODEL (PIT vs TTC) — transparan, bukan disembunyikan
# ==============================
with st.expander("⚠️ Limitasi Model: PIT vs TTC"):
    st.markdown("""
    Model PD di simulator ini bersifat **TTC (through-the-cycle)** — dilatih dari data historis
    tanpa overlay variabel makroekonomi (GDP growth, unemployment rate, dsb).

    PD **PIT (point-in-time)** yang sepenuhnya akurat idealnya menggabungkan PD TTC dengan overlay
    makro forward-looking sesuai kondisi ekonomi saat keputusan diambil. Scope proyek ini belum
    mengimplementasikan overlay makro penuh — angka PD di atas adalah estimasi **TTC**, bukan PIT.

    **Catatan lain:** simulator ini menilai 1 pengajuan baru (belum ada riwayat pembayaran), jadi
    secara definisi berada di **Stage 1 IFRS9** (belum ada SICR). Staging Stage 2/3 hanya relevan
    untuk portofolio loan yang sudah berjalan — lihat notebook final untuk staging portofolio historis.
    """)

# ==============================
# LGD SENSITIVITY
# ==============================
st.markdown("## 🔍 LGD Sensitivity")

lgd_range = np.linspace(0.0, 1.0, 50)
profits = [(1 - pd_lifetime) * interest_income - (pd_lifetime * l * EAD) for l in lgd_range]
chart_df = pd.DataFrame({"LGD": lgd_range, "Profit": profits})
st.line_chart(chart_df.set_index("LGD"))

# ==============================
# BREAK EVEN
# ==============================
st.markdown("## 🎯 Break-even LGD")

break_even = None
for l in lgd_range:
    profit = (1 - pd_lifetime) * interest_income - (pd_lifetime * l * EAD)
    if profit <= 0:
        break_even = l
        break

if break_even:
    st.info(f"Break-even LGD ≈ {break_even:.2f}")
else:
    st.success("Selalu profitable di rentang LGD 0-1.")

st.markdown("---")
st.caption("ML + Business + Decision Engine — IDx Partners Credit Risk Simulator")


Writing app.py


In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("***********************************")  

!streamlit run app.py &>/content/logs.txt &
public_url = ngrok.connect(8501)
print(public_url)